## Libraries

In [ ]:
import os, time
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim
from math import pi

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

## Dataset

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print("Device:", device)

nx = ny = 150
N_point = nx * ny
N_t = 77
L_left = 0.0
L_right = 4380.0
dt_phys = 1.0/3.0

trunc_spatial = 30
stride_x = 3
stride_y = 3
batch_size = 5000
epochs = 5001
lambda_pde = 1.0

D0_scale = 1.0
r_scale = 1.0
K_scale = 1.0

mat = sio.loadmat('/kaggle/input/dataset1/U_total_1.mat')
found = None
for k, v in mat.items():
    if isinstance(v, np.ndarray) and v.size >= N_point * N_t:
        found = v
        break
if found is None:
    raise RuntimeError("Couldn't find array")

U_flat = found.reshape(-1)[: N_point * N_t]
U_flat = torch.tensor(U_flat.reshape(-1,1), dtype=dtype, device=device)

X_total = torch.zeros((N_point * N_t, 1), dtype=dtype, device=device)
Y_total = torch.zeros_like(X_total)
for i in range(N_t):
    x = np.linspace(L_left, L_right, nx)
    y = np.linspace(L_left, L_right, ny)
    X, Y = np.meshgrid(x, y)
    X_total[i*N_point:(i+1)*N_point] = torch.tensor(X.reshape(-1,1), dtype=dtype, device=device)
    Y_total[i*N_point:(i+1)*N_point] = torch.tensor(Y.reshape(-1,1), dtype=dtype, device=device)

t_total = torch.zeros((N_point * N_t, 1), dtype=dtype, device=device)
for i in range(N_t):
    t_total[i*N_point:(i+1)*N_point] = (i+1) * dt_phys + 0.001

def normalize_0_1(tensor):
    mn = torch.min(tensor)
    mx = torch.max(tensor)
    return (tensor - mn) / (mx - mn + 1e-12), mn, mx

x_d_norm, x_min, x_max = normalize_0_1(X_total)
y_d_norm, y_min, y_max = normalize_0_1(Y_total)
t_d_norm, t_min, t_max = normalize_0_1(t_total)
u_d_norm, u_min, u_max = normalize_0_1(U_flat)

X_d = torch.cat([t_d_norm, x_d_norm, y_d_norm], dim=1)
U_norm = u_d_norm.clone()

x_phys_unique = np.unique(X_total.cpu().numpy())
y_phys_unique = np.unique(Y_total.cpu().numpy())
t_phys_unique = np.array([((i+1)*dt_phys + 0.001) for i in range(N_t)], dtype=np.float64)

def norm_phys_to_model_arr(arr, mn, mx):
    if not torch.is_tensor(arr):
        arr = torch.tensor(arr, dtype=torch.float32, device=device)
    if not torch.is_tensor(mn):
        mn = torch.tensor(mn, dtype=torch.float32, device=device)
    if not torch.is_tensor(mx):
        mx = torch.tensor(mx, dtype=torch.float32, device=device)
    return (arr - mn) / (mx - mn + 1e-12)

x_unique_norm = norm_phys_to_model_arr(x_phys_unique, x_min, x_max).reshape(-1,1)
y_unique_norm = norm_phys_to_model_arr(y_phys_unique, y_min, y_max).reshape(-1,1)
t_unique_norm = norm_phys_to_model_arr(t_phys_unique, t_min, t_max).reshape(-1,1)

## Fractional

In [ ]:
def fractional_weights_gl(alpha_tensor, N):
    k = torch.arange(0, N, dtype=dtype, device=device)
    lg1 = torch.lgamma(alpha_tensor + 1.0)
    lgk = torch.lgamma(k + 1.0)
    lg2 = torch.lgamma(alpha_tensor - k + 1.0)
    binom = torch.exp(lg1 - lgk - lg2)
    signs = ((-1.0)**k).to(dtype=dtype)
    return signs * binom

def Lalpha_GL_2D(u_grid, alpha_tensor, dx, dy, trunc=30):
    nx_local, ny_local, Nt_local = u_grid.shape
    w = fractional_weights_gl(alpha_tensor, trunc)
    pref_x = 1.0 / (dx ** alpha_tensor)
    pref_y = 1.0 / (dy ** alpha_tensor)
    out = torch.zeros_like(u_grid)
    for tt in range(Nt_local):
        slice_t = u_grid[:,:,tt]
        left_pad_x = slice_t[0:1,:].repeat(trunc,1)
        u_padded_x = torch.cat([left_pad_x, slice_t], dim=0)
        windows_x = torch.stack([u_padded_x[i:i+trunc,:] for i in range(nx_local)], dim=0)
        out_x = torch.tensordot(w, windows_x, dims=([0],[1])).transpose(0,1)
        left_pad_y = slice_t[:,0:1].repeat(1,trunc)
        u_padded_y = torch.cat([left_pad_y, slice_t], dim=1)
        windows_y = torch.stack([u_padded_y[:,i:i+trunc] for i in range(ny_local)], dim=1)
        out_y = torch.tensordot(w, windows_y, dims=([0],[2]))
        out[:,:,tt] = out_x * pref_x + out_y * pref_y
    return out

def build_L1_matrix(Nt_local, gamma_tensor, dt_local):
    if Nt_local <= 1:
        return torch.zeros((Nt_local, max(1,Nt_local-1)), dtype=dtype, device=device), torch.tensor(0.0, device=device)
    idx = torch.arange(1, Nt_local, dtype=dtype, device=device)
    b = torch.pow(idx, 1.0 - gamma_tensor) - torch.pow(idx - 1.0, 1.0 - gamma_tensor)
    M = torch.zeros((Nt_local, Nt_local-1), dtype=dtype, device=device)
    for n in range(1, Nt_local):
        seq = torch.flip(b[:n], dims=[0])
        M[n, :n] = seq
    C = 1.0 / (torch.exp(torch.lgamma(2.0 - gamma_tensor)) * (dt_local ** gamma_tensor))
    return M, C


## PINN

In [ ]:
class ParameterEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        self.raw_D0    = nn.Parameter(torch.tensor(0.0, dtype=dtype))  
        self.raw_r     = nn.Parameter(torch.tensor(0.0, dtype=dtype)) 
        self.raw_K     = nn.Parameter(torch.tensor(0.0, dtype=dtype))  
        self.raw_alpha = nn.Parameter(torch.tensor(0.5, dtype=dtype))  
        self.raw_gamma = nn.Parameter(torch.tensor(0.5, dtype=dtype)) 
    def D0(self):
        return 0.5 + (2.0 - 0.5) * torch.sigmoid(self.raw_D0)

    def r(self):
        return 0.01 + (1.0 - 0.01) * torch.sigmoid(self.raw_r)

    def K(self):
        return 2.5 + (5.0 - 2.5) * torch.sigmoid(self.raw_K)

    def alpha(self):
        return 0.02 + 1.98 * torch.sigmoid(self.raw_alpha)

    def gamma(self):
        return 0.01 + 0.99 * torch.sigmoid(self.raw_gamma)


param_est = ParameterEstimator().to(device)

class FCN3D(nn.Module):
    def __init__(self, N_INPUT=3, N_OUTPUT=1, N_HIDDEN=40, N_LAYERS=3):
        super().__init__()
        act = nn.Tanh
        layers = [nn.Linear(N_INPUT, N_HIDDEN), act()]
        for _ in range(N_LAYERS-1):
            layers += [nn.Linear(N_HIDDEN, N_HIDDEN), act()]
        layers += [nn.Linear(N_HIDDEN, N_OUTPUT)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

pinn = FCN3D().to(device)

def predict_on_grid_batched(pinn, x_u, y_u, t_u, batch_size_pred=5000):
    xv = x_u.view(-1)
    yv = y_u.view(-1)
    tv = t_u.view(-1)
    coords_list = []
    for tt in tv:
        tx = tt.repeat(xv.shape[0]*yv.shape[0])
        xx = xv.repeat_interleave(yv.shape[0])
        yy = yv.repeat(xv.shape[0])
        coords_list.append(torch.stack([tx,xx,yy],dim=1))
    coords_all = torch.cat(coords_list,dim=0)
    out_vals = []
    for start in range(0, coords_all.shape[0], batch_size_pred):
        end = min(start + batch_size_pred, coords_all.shape[0])
        batch = coords_all[start:end]
        out_vals.append(pinn(batch).view(-1))
    out_flat = torch.cat(out_vals, dim=0)
    nt, nx_, ny_ = tv.shape[0], xv.shape[0], yv.shape[0]
    u_np = out_flat.view(nt, nx_, ny_).permute(1,2,0)
    return u_np

x_coarse_norm = x_unique_norm[::stride_x]
y_coarse_norm = y_unique_norm[::stride_y]
t_coarse_norm = t_unique_norm
dx_phys = float(x_phys_unique[1]-x_phys_unique[0])
dy_phys = float(y_phys_unique[1]-y_phys_unique[0])

optimizer = optim.Adam([{'params': pinn.parameters(), 'lr': 1e-3},{'params': param_est.parameters(), 'lr': 5e-3}])

N_total = X_d.shape[0]
loss_history = []
start_time = time.time()

for epoch in range(epochs):
    idx = torch.randint(0, N_total, (min(batch_size, N_total),), device=device)
    Xb = X_d[idx]
    Ub_true = U_norm[idx]
    upred = pinn(Xb)
    data_loss = torch.mean((upred - Ub_true)**2)

    u_grid = predict_on_grid_batched(pinn, x_coarse_norm, y_coarse_norm, t_coarse_norm)
    nx_c, ny_c, Nt_c = u_grid.shape
    Npos = nx_c * ny_c
    u_mat = u_grid.reshape(Npos, Nt_c)

    if Nt_c > 1:
        diffs = u_mat[:,1:] - u_mat[:,:-1]
    else:
        diffs = torch.zeros((Npos, 1), device=device)

    gamma_tensor = param_est.gamma()
    M, C = build_L1_matrix(Nt_c, gamma_tensor, dt_phys)
    if Nt_c > 1:
        Lgamma_mat = C * (diffs @ M.T)
    else:
        Lgamma_mat = torch.zeros((Npos, Nt_c), device=device)
    Lgamma_mat[:,0] = 0.0
    Lgamma_3d = Lgamma_mat.reshape(nx_c, ny_c, Nt_c)

    alpha_tensor = param_est.alpha()
    Lalpha_3d = Lalpha_GL_2D(u_grid, alpha_tensor, dx_phys, dy_phys, trunc=trunc_spatial)

    D0_phys = param_est.D0() * D0_scale
    r_phys  = param_est.r()  * r_scale
    K_phys  = param_est.K() * K_scale

    residual = Lgamma_3d + D0_phys * Lalpha_3d - r_phys * u_grid * (1.0 - u_grid / K_phys)
    pde_loss = torch.mean(residual[:,:,1:] ** 2) if residual.shape[2] > 1 else torch.mean(residual**2)

    total_loss = data_loss + lambda_pde * pde_loss

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    loss_history.append(total_loss.item())

    if epoch % 100 == 0:
        elapsed = time.time() - start_time
        print(f"Epoch {epoch:5d}, "f"Total={total_loss.item():.4e}, Data={data_loss.item():.4e}, PDE={pde_loss.item():.4e}")


print("Final parameters values:")
print(f"D0 = {param_est.D0().item()*D0_scale:.6e}")
print(f"r  = {param_est.r().item()*r_scale:.6e}")
print(f"K  = {param_est.K().item()*K_scale:.6e}")
print(f"alpha = {param_est.alpha().item():.6e}")
print(f"gamma = {param_est.gamma().item():.6e}")
